# Quantum Reservoir Computing Demo (Python)

In this notebook we classify MNIST images using **Quantum Reservoir Computing (QRC)** in Python. Features extracted from MNIST images are mapped into a high-dimensional space via the chaotic dynamics of a simulated quantum system.

This demo uses classical numerical simulation of a neutral-atom Rydberg quantum system with 8 qubits, using the **local detuning** encoding strategy.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import expm
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import accuracy_score
from tqdm.auto import tqdm
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)
print("Libraries loaded.")


## The MNIST Dataset
MNIST contains 28×28 grayscale images of handwritten digits (0–9). We use 10,000 training and 1,000 test samples.

In [ ]:
try:
    from tensorflow.keras.datasets import mnist
    (X_tr_raw, y_tr_raw), (X_te_raw, y_te_raw) = mnist.load_data()
    X_tr_raw = X_tr_raw.reshape(-1, 784).astype(np.float32) / 255.0
    X_te_raw = X_te_raw.reshape(-1, 784).astype(np.float32) / 255.0
except ImportError:
    from sklearn.datasets import fetch_openml
    print("Fetching MNIST via OpenML...")
    data = fetch_openml('mnist_784', version=1, as_frame=False, parser='auto')
    X_all = data.data.astype(np.float32) / 255.0
    y_all = data.target.astype(int)
    X_tr_raw, X_te_raw = X_all[:60000], X_all[60000:]
    y_tr_raw, y_te_raw = y_all[:60000], y_all[60000:]

NUM_TRAIN, NUM_TEST = 10000, 1000
X_train_flat, y_train = X_tr_raw[:NUM_TRAIN], y_tr_raw[:NUM_TRAIN]
X_test_flat,  y_test  = X_te_raw[:NUM_TEST],  y_te_raw[:NUM_TEST]
print(f"Train: {X_train_flat.shape}, Test: {X_test_flat.shape}")


In [ ]:
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
for idx in range(20):
    ax = axes[idx // 10, idx % 10]
    ax.imshow(X_train_flat[idx].reshape(28, 28), cmap='gray')
    ax.set_title(str(y_train[idx]), fontsize=10)
    ax.axis('off')
plt.suptitle("Sample MNIST Images", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()


## PCA Dimensionality Reduction

We use PCA to reduce each 784-pixel image to **8 principal components**, which are then scaled to the detuning range $[-6, 6]$ rad/µs. Each component is mapped to one atom's local detuning.

In [ ]:
DIM_PCA = 8
pca = PCA(n_components=DIM_PCA)
X_pca_train = pca.fit_transform(X_train_flat)
X_pca_test  = pca.transform(X_test_flat)

# Plot explained variance
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.bar(range(1, DIM_PCA+1), pca.explained_variance_ratio_ * 100)
ax1.set_xlabel("Principal Component")
ax1.set_ylabel("Explained Variance (%)")
ax1.set_title("PCA Explained Variance")

DELTA_MAX = 6.0
spectral = np.max(np.abs(X_pca_train))
X_qrc_train = X_pca_train / spectral * DELTA_MAX
X_qrc_test  = X_pca_test  / spectral * DELTA_MAX

sc = ax2.scatter(X_qrc_train[:500, 0], X_qrc_train[:500, 1],
                 c=y_train[:500], cmap='tab10', alpha=0.5, s=10)
plt.colorbar(sc, ax=ax2, label='Digit')
ax2.set_xlabel("PC 1 (scaled)"); ax2.set_ylabel("PC 2 (scaled)")
ax2.set_title("First Two PCA Components (scaled detunings)")
plt.tight_layout(); plt.show()
print(f"PCA features range: [{X_qrc_train.min():.2f}, {X_qrc_train.max():.2f}] rad/µs")


## Simulating Quantum Dynamics — The Rydberg Hamiltonian

We simulate a chain of 8 neutral atoms interacting via the Rydberg Hamiltonian:

$$H = \frac{\Omega}{2}\sum_i \sigma_i^x - \sum_i \Delta_i n_i + \sum_{i<j} V_{ij} n_i n_j$$

where:
- $\Omega = 2\pi$ rad/µs — global Rabi frequency (drive)
- $\Delta_i$ — local detuning on atom $i$, **set to the scaled PCA components**
- $n_i = |r\rangle\langle r|_i = (I - \sigma_z^i)/2$ — Rydberg excitation number operator
- $V_{ij} = C_6 / |r_i - r_j|^6$ — van der Waals interaction

Starting from all atoms in the ground state $|g\cdots g\rangle$, we evolve under this Hamiltonian and measure $\langle\sigma_z^i\rangle$ and $\langle\sigma_z^i \sigma_z^j\rangle$ at $t = 0.5, 1.0, \ldots, 4.0$ µs, yielding a **288-dimensional embedding** for each image.

In [ ]:
# ── Pauli / number operators ──────────────────────────────────────────────────
sx = np.array([[0,1],[1,0]], dtype=complex)
sz = np.array([[1,0],[0,-1]], dtype=complex)
n_op = (np.eye(2, dtype=complex) - sz) / 2   # |r><r| = excited-state projector

def kron_op(n_qubits, op, site):
    'Embed single-qubit op at site into n_qubits Hilbert space.'
    ops = [np.eye(2, dtype=complex)] * n_qubits
    ops[site] = op
    out = ops[0]
    for o in ops[1:]:
        out = np.kron(out, o)
    return out

def build_rydberg_hamiltonian(nsites, Delta, Omega, Vmat):
    dim = 2**nsites
    H = np.zeros((dim, dim), dtype=complex)
    for i in range(nsites):
        H += (Omega / 2) * kron_op(nsites, sx, i)
        H -= Delta[i]    * kron_op(nsites, n_op, i)
    for i in range(nsites):
        for j in range(i+1, nsites):
            if Vmat[i, j] != 0:
                H += Vmat[i, j] * (kron_op(nsites, n_op, i) @ kron_op(nsites, n_op, j))
    return H

# ── Interaction matrix ────────────────────────────────────────────────────────
def generate_Vmat(locs, C6):
    nsites = len(locs)
    Vmat = np.zeros((nsites, nsites))
    for i in range(nsites):
        for j in range(nsites):
            if i != j:
                d = np.linalg.norm(locs[i] - locs[j])
                Vmat[i, j] = C6 / d**6
    return Vmat

print("Quantum operators defined.")


In [ ]:
def apply_qrc_layer_single(Delta, nsites, locs, Omega, C6, t_start, t_end, step):
    Vmat = generate_Vmat(locs, C6)
    H = build_rydberg_hamiltonian(nsites, Delta, Omega, Vmat)

    # One-step propagator (Hamiltonian is constant for this image)
    U = expm(-1j * H * step)

    psi = np.zeros(2**nsites, dtype=complex)
    psi[0] = 1.0   # |g...g> = |00...0> = index 0

    steps = int(round((t_end - t_start) / step))
    Z_ops  = [kron_op(nsites, sz, i) for i in range(nsites)]
    ZZ_ops = [(i, j, Z_ops[i] @ Z_ops[j])
              for i in range(nsites) for j in range(i+1, nsites)]

    out_z, out_zz = [], []
    for _ in range(steps):
        psi = U @ psi
        rho = np.outer(psi, psi.conj())   # density matrix (pure state)
        for Zi in Z_ops:
            out_z.append(np.real(np.trace(rho @ Zi)))
        for _, _, ZiZj in ZZ_ops:
            out_zz.append(np.real(np.trace(rho @ ZiZj)))

    return np.array(out_z + out_zz)    # length = steps*(nsites + npairs)

def apply_qrc_batch(X_features, nsites, locs, Omega, C6, t_start, t_end, step):
    outs = []
    for i in tqdm(range(len(X_features)), desc="QRC Simulation"):
        outs.append(apply_qrc_layer_single(
            X_features[i], nsites, locs, Omega, C6, t_start, t_end, step))
    return np.array(outs)

print("QRC layer functions defined.")


## Configuring the Reservoir

We place **8 atoms in a chain** with 10 µm spacing. The key **tunable parameters** are:
| Parameter | Symbol | Default | Effect |
|-----------|--------|---------|--------|
| Rabi freq | $\Omega$ | $2\pi$ rad/µs | Drive strength |
| Atom spacing | $d$ | 10 µm | Scales all $V_{ij}$ interactions |
| C6 coefficient | $C_6$ | 862690·2π | Interaction strength |
| Evolution time | $t_{end}$ | 4.0 µs | Reservoir memory depth |
| Readout step | step | 0.5 µs | Temporal resolution |

Try changing `ATOM_SPACING`, `T_END`, or `C6` and re-run the cells below!

In [ ]:
# ── Tunable Reservoir Parameters ─────────────────────────────────────────────
NSITES       = DIM_PCA          # 8 atoms = 8 PCA components
OMEGA        = 2 * np.pi        # Rabi frequency (rad/µs)
ATOM_SPACING = 10.0             # µm  ← change me!
C6           = 862690 * 2*np.pi # rad/µs·µm^6  ← change me!
T_START      = 0.0
T_END        = 4.0              # µs  ← change me!
STEP         = 0.5              # readout every 0.5 µs

locs = np.array([[i * ATOM_SPACING, 0.0] for i in range(NSITES)])

steps_total = int(round((T_END - T_START) / STEP))
npairs = NSITES * (NSITES - 1) // 2
emb_dim = steps_total * (NSITES + npairs)
print(f"Reservoir config: {NSITES} atoms, {steps_total} time steps → {emb_dim}-dim embeddings")


In [ ]:
print("Generating QRC training embeddings...")
embeddings_train = apply_qrc_batch(X_qrc_train, NSITES, locs, OMEGA, C6, T_START, T_END, STEP)
print(f"Train embeddings: {embeddings_train.shape}")


In [ ]:
print("Generating QRC test embeddings...")
embeddings_test = apply_qrc_batch(X_qrc_test, NSITES, locs, OMEGA, C6, T_START, T_END, STEP)
print(f"Test embeddings: {embeddings_test.shape}")


## Training and Evaluation
We compare three classifiers: a linear SVM on raw PCA features, a linear SVM on QRC embeddings, and a nonlinear neural network on raw PCA features.

In [ ]:
def train_svm(X_tr, y_tr, X_te, y_te, label):
    svm = SVC(kernel='linear', max_iter=5000)
    svm.fit(X_tr, y_tr)
    tr_acc = accuracy_score(y_tr, svm.predict(X_tr))
    te_acc = accuracy_score(y_te, svm.predict(X_te))
    print(f"[{label}] Train={tr_acc:.3f}  Test={te_acc:.3f}")
    return tr_acc, te_acc

tr_pca, te_pca = train_svm(X_qrc_train, y_train, X_qrc_test, y_test, "PCA + Linear SVM (baseline)")
tr_qrc, te_qrc = train_svm(embeddings_train, y_train, embeddings_test, y_test, "QRC + Linear SVM")


In [ ]:
# Nonlinear NN baseline on raw PCA
mlp = MLPClassifier(hidden_layer_sizes=(100, 100), max_iter=200, random_state=42)
mlp.fit(X_qrc_train, y_train)
tr_nn = accuracy_score(y_train, mlp.predict(X_qrc_train))
te_nn = accuracy_score(y_test,  mlp.predict(X_qrc_test))
print(f"[PCA + MLP (2-hidden layers)] Train={tr_nn:.3f}  Test={te_nn:.3f}")


In [ ]:
labels = ['PCA\n(Linear SVM)', 'QRC\n(Linear SVM)', 'PCA\n(Neural Net)']
train_accs = [tr_pca, tr_qrc, tr_nn]
test_accs  = [te_pca, te_qrc, te_nn]

x = np.arange(len(labels))
w = 0.35
fig, ax = plt.subplots(figsize=(7, 5))
bars1 = ax.bar(x - w/2, train_accs, w, label='Train', color='steelblue')
bars2 = ax.bar(x + w/2, test_accs,  w, label='Test',  color='coral')
ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=11)
ax.set_ylabel("Accuracy"); ax.set_ylim(0, 1.05)
ax.set_title("QRC vs. Baseline Classifiers (MNIST)")
ax.legend()
for bar in list(bars1) + list(bars2):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
            f"{bar.get_height():.2f}", ha='center', va='bottom', fontsize=9)
plt.tight_layout(); plt.show()


## Parameter Sweep — Optimize Reservoir Performance

Sweep over `T_END` and `ATOM_SPACING` to find the best configuration.

In [ ]:
# Quick sweep (small subset for speed)
SWEEP_N = 500
sweep_accs = {}

for t_end in [2.0, 4.0, 6.0]:
    layer_cfg = dict(nsites=NSITES, locs=locs, Omega=OMEGA, C6=C6,
                     t_start=T_START, t_end=t_end, step=STEP)
    emb_tr = apply_qrc_batch(X_qrc_train[:SWEEP_N], **layer_cfg)
    emb_te = apply_qrc_batch(X_qrc_test[:SWEEP_N],  **layer_cfg)
    svm = SVC(kernel='linear', max_iter=3000).fit(emb_tr, y_train[:SWEEP_N])
    acc = accuracy_score(y_test[:SWEEP_N], svm.predict(emb_te))
    sweep_accs[f"t_end={t_end}"] = acc
    print(f"t_end={t_end}: test acc={acc:.3f}")

plt.figure(figsize=(6, 4))
plt.bar(list(sweep_accs.keys()), list(sweep_accs.values()), color='teal')
plt.ylabel("Test Accuracy"); plt.title("Accuracy vs. Evolution Time (T_end)")
plt.ylim(0, 1); plt.tight_layout(); plt.show()


## Summary

| Method | Test Accuracy |
|--------|--------------|
| PCA + Linear SVM (baseline) | — |
| QRC + Linear SVM | — |
| PCA + Neural Net | — |

The QRC embedding allows a simple linear classifier to match or exceed a nonlinear neural network, demonstrating the power of quantum dynamics as a non-linear feature map.

**Key insight:** The reservoir's nonlinear interaction dynamics create a rich feature space without requiring gradient-based training of the quantum system.